# Sesión 10 · Cuantiles

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

## 1 · Los datos

In [ ]:
library(tidyverse)

url <- paste0("https://docs.google.com/spreadsheets/d/e/2PACX-1vTTKC57eWZVIQ9lwhoR5nV",
              "qYM4kgi5zA9yifa-YStdfdJwNe7ATs0p-TUCUwvjfcWmHmvsZDEK8VX4I/pub?gid=1326008067&single=true&output=csv")

grupos <- read_csv(url, show_col_types = FALSE) |>
  select(
    carrera  = Carrera,
    genero   = `Género`,
    estatura = `Estatura, en metros`,
    traslado = `Tiempo de traslado a la universidad, en minutos`
  ) |>
  filter(!is.na(traslado), !is.na(estatura))

glimpse(grupos)

## 2 · Primero a mano

Estos son los tiempos de once compañeros, ordenados de menor a mayor.

Antes de calcular: ¿qué valor parte esta lista en dos mitades?

In [ ]:
once <- sort(head(grupos$traslado, 11))
once

Un cuartil se localiza primero por su **posición** en la lista ordenada:

$$Q_k = \frac{k\,(n+1)}{4}$$

Con 11 datos, el primer cuartil está en la posición $\frac{1 \times 12}{4} = 3$,
o sea el tercer dato.

In [ ]:
n <- length(once)

(n + 1) * 0.25      # la posición
once[3]             # el dato que está en esa posición

In [ ]:
# Los tres cortes de una vez
posiciones <- (n + 1) * c(0.25, 0.50, 0.75)
posiciones

once[posiciones]

In [ ]:
# Y la función que hace lo mismo
quantile(once, probs = c(0.25, 0.50, 0.75))

### ¿Coinciden siempre?

Con once datos las posiciones cayeron en un dato exacto. Con el grupo completo la
posición cae entre dos datos y el cuantil se interpola.

In [ ]:
estatura <- sort(grupos$estatura)
m <- length(estatura)

(m + 1) * 0.75      # ya no es un entero

In [ ]:
# A mano, interpolando entre los dos datos vecinos
pos <- (m + 1) * 0.75
bajo <- floor(pos)

estatura[bajo] + (pos - bajo) * (estatura[bajo + 1] - estatura[bajo])

In [ ]:
# Y lo que devuelve R
quantile(grupos$estatura, 0.75)

**No coinciden.** R tiene nueve definiciones de cuantil. La fórmula $(n+1)p$ que
usamos a mano es el tipo 6; la que R usa por omisión es el tipo 7.

In [ ]:
x <- c(12, 14, 15, 18, 19, 22, 25, 28, 31, 40)

sapply(1:9, function(t) quantile(x, 0.25, type = t))

En Excel pasa lo mismo: `PERCENTIL.INC` es el tipo 7 y `PERCENTIL.EXC` es el
tipo 6.

Si le preguntas a un modelo de lenguaje cuál es el primer cuartil, vas a recibir
un número sin saber con qué convención se calculó. La pregunta útil es cuál de
las nueve definiciones usó, y si es la misma que usa el resto de tu trabajo.

## 3 · Cuartiles, mediana y rango intercuartil

In [ ]:
quantile(grupos$traslado, probs = c(0.25, 0.50, 0.75))

El segundo cuartil es la **mediana**: parte al grupo en dos mitades del mismo
tamaño.

$$Q_2 = D_5 = P_{50}$$

El **rango intercuartil** es la distancia entre el primer y el tercer cuartil, o
sea el tramo donde vive la mitad central:

$$RIC = Q_3 - Q_1$$

In [ ]:
q <- quantile(grupos$traslado, c(0.25, 0.75))
q[2] - q[1]

IQR(grupos$traslado)

## 4 · Deciles

Nueve cortes. `seq()` los genera sin escribirlos uno por uno.

In [ ]:
seq(0.1, 0.9, by = 0.1)

In [ ]:
quantile(grupos$traslado, probs = seq(0.1, 0.9, by = 0.1))

### Los deciles de ingreso de México

Antes de correr la celda: pensando en el ingreso mensual de tu hogar, ¿en cuál de
los diez deciles crees que cae?

La tabla viene de la Encuesta Nacional de Ingresos y Gastos de los Hogares
(ENIGH) 2024 del INEGI. El ingreso se publica trimestral; aquí está dividido
entre tres.

In [ ]:
deciles <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/enigh2024_deciles.csv", show_col_types = FALSE)

deciles |>
  transmute(
    decil,
    ingreso_mensual = round(mensual),
    hasta           = round(tope_mensual),
    porcentaje_del_ingreso = round(porcentaje_del_ingreso, 1)
  )

El decil dice en qué lugar del reparto cae un hogar. No dice si ese ingreso
alcanza: eso depende de cuántas personas viven de él, de dónde viven y de qué
cuestan ahí las cosas.

El decil X incluye tanto al hogar que recibe 50 mil pesos al mes como al que
recibe millones. Un corte por posición no describe lo que pasa dentro del último
tramo.

## 5 · Percentiles

Cien partes, para cuando las diez de los deciles resultan gruesas.

In [ ]:
quantile(grupos$traslado, probs = c(0.05, 0.90, 0.95, 0.99))

Con este número de respuestas, el percentil 37 y el 38 casi siempre caen entre
los dos mismos datos. El resultado sale con decimales y eso no significa que esté
medido con esa precisión.

## 6 · El resumen de cinco números

In [ ]:
quantile(grupos$traslado, probs = seq(0, 1, by = 0.25))

Mínimo, $Q_1$, mediana, $Q_3$ y máximo describen la distribución completa con
cinco valores. Son los que dibuja el diagrama de caja de la próxima sesión.

## 7 · Tu percentil

Cambia el número por tu propio tiempo de traslado.

In [ ]:
mi_traslado <- 30

mean(grupos$traslado <= mi_traslado) * 100

## Tarea

Sobre **tu serie del portafolio**:

1. Los cuartiles de tu variable numérica, y una oración que interprete $Q_1$ y
   $Q_3$ con las unidades de tu serie.
2. El rango intercuartil, y qué tramo de tus datos describe.
3. Un percentil que sirva para decidir algo en tu dominio, elegido por ti, con la
   razón de por qué ese y no otro.
4. Una línea sobre qué no dice ese percentil.

**Guarda tu copia.**